## Ch7-01 — Symbolic energy binding

This notebook introduces sympy symbolic binding for the `DeliveredEnergy` calc def; after running it you can verify the 67200 J reference value and compare the symbolic expression with the SysML formula.


Chapter 3 introduced `DeliveredEnergy` as a `calc def` with the formula `power * duration * efficiency`. This notebook binds that same formula to a sympy expression and evaluates it with lambdify, establishing the reference value (67200 J) that the parameter sweep in notebook 03 builds on. See [Ch3-02 MoP candidate evaluation](../ch03-measures/02-mop-candidate-eval.ipynb) for the original calc def.


In [ ]:
from pathlib import Path
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")
source = Path("../../models/ch07-cumulative.sysml").read_text()
print(source)
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"

The `ch07-cumulative.sysml` file adds `state Cycle` with four substates (`idle`, `heating`, `ready`, `cancelled`) and three transitions (`idle → heating` on `Start`, `heating → ready` on `Finish`, `heating → cancelled` on `Cancel`). This is construct 13 — the first executable behavior in the model. `model.execute_state()` can trace event sequences through this state machine.

In [ ]:
# A calc def referencing an undefined base type fails to parse.
bad_source = """
package P {
    calc def Broken :> MissingBase {
        in x : Real;
        return : Real = x;
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok, "Expected parse failure for undefined base type"
# Expected: diagnostic pointing to 'MissingBase' as an unresolved reference
print(f"Negative control ok: bad.ok={bad.ok}")


SysML v2's `calc def` expresses `power * duration * efficiency` as a formula in the model. Sympy lets Python work with the same formula as a symbolic expression: `sp.symbols('P t eta', positive=True)` creates three variables that know they are positive quantities, matching the `in` parameters of `DeliveredEnergy`. Jupyter renders a sympy expression as typeset math — the cell below shows what the formula looks like before any numbers go in.


In [ ]:
import sympy as sp

P, t, eta = sp.symbols('P t eta', positive=True)
Q_sym = P * t * eta    # mirrors: return : Real = power * duration * efficiency
Q_sym                  # Jupyter renders this as typeset math


In [ ]:
# lambdify compiles Q_sym into a numpy-compatible function.
# [P, t, eta] fixes the argument order to match the calc def's in-parameters.
Q_fn = sp.lambdify([P, t, eta], Q_sym, 'numpy')

# Reference value: 800 W × 120 s × 0.7 = 67200 J
ref = float(Q_fn(800.0, 120.0, 0.7))
assert abs(ref - 67200.0) < 1.0, f"Reference mismatch: {ref}"
print(f"Q_fn(800, 120, 0.7) = {ref:.1f} J  (expected 67200.0)")


`model.eval()` evaluates an expression through the opensysml runtime using the model's own attribute values. Passing a qualified call string invokes the calc def directly through the model, independent of the sympy binding. The two results should agree to within floating-point tolerance, confirming that the sympy expression faithfully mirrors the model formula.


In [ ]:
model_val = float(model.eval("ToasterDemo::DeliveredEnergy(800.0, 120.0, 0.7)"))
assert abs(model_val - 67200.0) < 1.0, f"Model eval mismatch: {model_val}"
print(f"model.eval(...) = {model_val:.1f} J")
print(f"Both agree: {abs(ref - model_val) < 1.0}")
conn.close()


The `calc def DeliveredEnergy` with formula `power * duration * efficiency` (A-F) is bound to a sympy expression and evaluated by lambdify (O-S); `Q_fn(800.0, 120.0, 0.7)` returns 67200.0, matching the `model.eval()` reference value (E).


Try the chapter exercise in `exercises/ch07/exercise.ipynb`: bind the coffee maker's brew energy formula to sympy and verify the reference value for a 1200 W heating element running for 90 seconds at 0.65 efficiency.
